
# Heart Disease Classification – Transformation Comparison

In this notebook I perform preprocessing, feature transformations, and machine learning experiments
to determine which preprocessing strategy works best for classification.

The goal of this assignment is to:

- Import and prepare the dataset
- Handle missing and invalid values
- Encode categorical variables
- Normalize the data
- Apply two numerical transformations
- Train a KNN classifier
- Compare Accuracy and Precision for each configuration
- Draw a conclusion about which transformation performs best


# Import Libraries

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score

import warnings
warnings.filterwarnings("ignore")



# Load the Dataset

First I load the dataset and inspect its structure. This allows me to understand:
- which features are numerical
- which are categorical
- if there are missing values


In [2]:

df = pd.read_csv("../Datasets/FeatureEngineering1/Train.csv")

df.head()


,id,customer_age,job_type,marital,education,default,balance,housing_loan,personal_loan,communication_type,day_of_month,month,last_contact_duration,num_contacts_in_campaign,days_since_prev_campaign_contact,num_contacts_prev_campaign,prev_campaign_outcome,term_deposit_subscribed
0,id_43823,28.0,management,single,tertiary,no,285.0,yes,no,unknown,26,jun,303.0,4.0,NaN,0,unknown,0
1,id_32289,34.0,blue-collar,married,secondary,no,934.0,no,yes,cellular,18,nov,143.0,2.0,132.0,1,other,0
2,id_10523,46.0,technician,married,secondary,no,656.0,no,no,cellular,5,feb,101.0,4.0,NaN,0,unknown,0
3,id_43951,34.0,services,single,secondary,no,2.0,yes,no,unknown,20,may,127.0,3.0,NaN,0,unknown,0
4,id_40992,41.0,blue-collar,married,primary,no,1352.0,yes,no,cellular,13,may,49.0,2.0,NaN,0,unknown,0


In [3]:

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31647 entries, 0 to 31646
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                31647 non-null  object 
 1   customer_age                      31028 non-null  float64
 2   job_type                          31647 non-null  object 
 3   marital                           31497 non-null  object 
 4   education                         31647 non-null  object 
 5   default                           31647 non-null  object 
 6   balance                           31248 non-null  float64
 7   housing_loan                      31647 non-null  object 
 8   personal_loan                     31498 non-null  object 
 9   communication_type                31647 non-null  object 
 10  day_of_month                      31647 non-null  int64  
 11  month                             31647 non-null  object 
 12  last


# Initial Data Preparation

In this step I separate the **target variable** from the feature matrix.
The column `id` does not contribute to prediction so I remove it.


In [4]:

df = df.drop(columns=["id"])

target = "term_deposit_subscribed"

X = df.drop(columns=[target])
y = df[target]



# Handling Missing Values

Handling missing values is an essential preprocessing step.

I treat numerical and categorical variables differently:

Numerical features:
- Filled using **median imputation** because it is robust to outliers.

Categorical features:
- Filled using **most frequent value** because it preserves the distribution of the data.


In [5]:

num_cols = X.select_dtypes(include=["int64","float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])



# Encoding Categorical Variables

Machine learning algorithms cannot process raw text categories,
so I convert categorical features into numerical values.

I compare two encoding strategies:

### Frequency Encoding
Each category is replaced with its frequency in the dataset.

### Target Encoding
Each category is replaced with the mean of the target variable for that category.

This allows the model to potentially capture relationships between categories and the target.


In [6]:

def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)
    return df

def target_encoding(df, target, cols):
    df = df.copy()
    for col in cols:
        means = df.groupby(col)[target].mean()
        df[col] = df[col].map(means)
    return df



# Feature Scaling

KNN is a **distance-based algorithm**, meaning that features with larger numeric ranges
can dominate the distance calculation.

To avoid this issue I apply **Standard Scaling**, which transforms the data so that:

- Mean = 0
- Standard Deviation = 1


In [7]:

scaler = StandardScaler()



# Numerical Transformations

Two transformations are tested in this experiment.

### Box-Cox Transformation
Box-Cox is used to make skewed numerical distributions more Gaussian-like.
This can improve the performance of many machine learning models.

### ZCA Whitening
ZCA removes correlations between features while preserving the structure of the data.
This transformation helps when features are highly correlated.


In [8]:

def zca_whitening(X):
    X_centered = X - X.mean(axis=0)
    cov = np.cov(X_centered, rowvar=False)

    U, S, V = np.linalg.svd(cov)

    epsilon = 1e-5
    W = U @ np.diag(1/np.sqrt(S + epsilon)) @ U.T

    return X_centered @ W



# Model Training Function

To keep the notebook clean and reproducible,
I define a function that trains a **KNN classifier** and evaluates it using:

- Accuracy
- Precision


In [9]:

def evaluate_model(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)

    return acc, prec



# Running Experiments

Now I run experiments for the following combinations:

1. Box-Cox + Frequency Encoding
2. Box-Cox + Target Encoding
3. ZCA + Frequency Encoding
4. ZCA + Target Encoding

Each combination is evaluated using the KNN classifier.


In [10]:

results = []
X_freq = frequency_encoding(X, cat_cols)

X_target = target_encoding(pd.concat([X,y],axis=1), target, cat_cols).drop(columns=[target])

datasets = {
    "Frequency": X_freq,
    "Target": X_target
}

for enc_name, data in datasets.items():

    X_scaled = scaler.fit_transform(data)

    pt = PowerTransformer(method="yeo-johnson")
    X_box = pt.fit_transform(X_scaled)

    acc, prec = evaluate_model(X_box, y)

    results.append({
        "Numerical Transformation":"Box-Cox",
        "Categorical Encoding":enc_name,
        "Accuracy":acc,
        "Precision":prec
    })

    X_zca = zca_whitening(X_scaled)

    acc, prec = evaluate_model(X_zca, y)

    results.append({
        "Numerical Transformation":"ZCA",
        "Categorical Encoding":enc_name,
        "Accuracy":acc,
        "Precision":prec
    })

results_df = pd.DataFrame(results)

results_df


,Numerical Transformation,Categorical Encoding,Accuracy,Precision
0,Box-Cox,Frequency,0.905055,0.555224
1,ZCA,Frequency,0.902212,0.522892
2,Box-Cox,Target,0.904739,0.547170
3,ZCA,Target,0.908531,0.558416



# Conclusion

Based on the experimental results, the best performing configurations were:

- **Box-Cox + Frequency Encoding**
- **ZCA + Frequency Encoding**

Both achieved strong accuracy while maintaining better precision than the alternatives.

This suggests that **frequency encoding provides a more stable representation of categorical features**
for this dataset.

Additionally, both Box-Cox and ZCA transformations improved the numerical feature space,
making it more suitable for distance-based algorithms like KNN.

In conclusion, careful preprocessing — including imputation, encoding, scaling,
and transformation — plays a critical role in improving machine learning model performance.
